# Реализация SSD
В это практическом уроке мы рассмотрим упрощённый пример реализации архитектуры SSD (Single Shot Multibox Detector). Цель урока -- разобраться с тем, как работает инференс в архитектуре SSD. Обучение SDD -- отдельный сложный вопрос, которы выходит за рамки данного урока.

### Загрузка необходимых библиотек
Здесь мы загружаем различне библиотеки, включая TensoFlow.


In [ ]:
import numpy as np

import tensorflow as tf
#tf.enable_eager_execution()
print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.7.0


### Модель SSD

Давайте реализуем класс, соответствующий модели SSD. Если предположить, что такая модель уже обучена, предсказание с её помощью сделать довольно просто.

Наша модель будет состоять из некоторого набора свёртчоных и пулинг слоёв, с помощью которых мы получаем несколько промежуточных карт признаков, соответствующих различным масштабам (`feat1`, `feat2`, `feat3`). Далее для каждого такого масштаба запускается детектор, который по сути является просто свёрточным слоем, задача которого предсказать классы и координаты боксов (всё это для каждого дефолт-бокса). Каждому пространственному пикселю тензора, который подаёдтся на вход в детектор соовтетствует несколько дефолт боксов (`num_def_boxes`), относительно которых мы и имщем объекты на картинке.

Итого, каждый детектор для одного пространственного пикселя карт признаков должен предсказать вектор размерности `N*(4+C)`, где N - кол-во дефолт боксов, C - кол-во классов.

Для наглядности можно раздеить каждый детектор на два параллельных свёрточных слоя: `conv_cls_i`, ответственный за классификацию дефолт-боксов (кол-во выходных каналов `N*C`), и `conv_loc_i`, ответственный за локализацию (кол-во выходных каналов `N*4`).

То, как устроены дефолт-боксы (их расположение и размеры) имеет значение во время обучения, но не нужно для инференса. Нам нужно лишь знать, сколько дефолт-боксов есть в нашей модели.

В конце соединим предсказания со всех детекторов.

**[ЗАДАНИЕ 1]** Вопрос: Какое максимальное количество боксов может предсказать такая модель, если размер входной картинки равен будет 128x128, а num_def_boxes=3?

In [ ]:
class SSD(tf.keras.Model):
    def __init__(self, num_classes, num_def_boxes):
        super().__init__()
        self.num_classes = num_classes

        # Слои для извлечения признаков
        self.conv1 = tf.keras.layers.Conv2D(32, (5, 5), activation=tf.nn.relu, padding='same')
        self.conv2 = tf.keras.layers.Conv2D(32, (5, 5), activation=tf.nn.relu, padding='same')
        self.conv3 = tf.keras.layers.Conv2D(64, (5, 5), activation=tf.nn.relu, padding='same')
        self.conv4 = tf.keras.layers.Conv2D(64, (5, 5), activation=tf.nn.relu, padding='same')
        self.conv5 = tf.keras.layers.Conv2D(128, (5, 5), activation=tf.nn.relu, padding='same')
        self.conv6 = tf.keras.layers.Conv2D(128, (5, 5), activation=tf.nn.relu, padding='same')

        # Классификационные части детекторов (отдельный детектор для каждого масштаба)
        # Для каждого пикселя карт признаков предсказываются
        # распределения вероятностей для всех дефолт боксов
        self.conv_cls1 = tf.keras.layers.Conv2D(num_def_boxes*num_classes, (3, 3), activation=tf.nn.relu, padding='same')
        self.conv_cls2 = tf.keras.layers.Conv2D(num_def_boxes*num_classes, (3, 3), activation=tf.nn.relu, padding='same')
        self.conv_cls3 = tf.keras.layers.Conv2D(num_def_boxes*num_classes, (3, 3), activation=tf.nn.relu, padding='same')

        # Локализационные части детекторов  (отдельный детектор для каждого масштаба)
        # Для каждого пикселя карт признаков предсказываются
        # координаты всех дефолт боксов
        self.conv_loc1 = tf.keras.layers.Conv2D(num_def_boxes*4, (3, 3), activation=tf.nn.relu, padding='same')
        self.conv_loc2 = tf.keras.layers.Conv2D(num_def_boxes*4, (3, 3), activation=tf.nn.relu, padding='same')
        self.conv_loc3 = tf.keras.layers.Conv2D(num_def_boxes*4, (3, 3), activation=tf.nn.relu, padding='same')

        self.pool = tf.keras.layers.MaxPooling2D((2, 2), (2, 2), padding='same')

    # Переход к тензору размера (batch, num_boxes, num_classes)
    # batch - кол-во образцов в батче
    # num_boxes - количество всех боксов для данного масштаба
    # num_classes - количество классов
    def reshape_cls(self, pred_cls):
        pred_cls = tf.transpose(pred_cls, (0, 3, 1, 2))
        pred_cls = tf.reshape(pred_cls, (pred_cls.shape[0], self.num_classes, -1))
        pred_cls = tf.transpose(pred_cls, (0, 2, 1))
        return pred_cls

    # Переход к тензору размера (batch, num_boxes, 4)
    # batch - кол-во образцов в батче
    # num_boxes - количество всех боксов для данного масштаба
    def reshape_loc(self, pred_loc):
        pred_loc = tf.transpose(pred_loc, (0, 3, 1, 2))
        pred_loc = tf.reshape(pred_loc, (pred_loc.shape[0], 4, -1))
        pred_loc = tf.transpose(pred_loc, (0, 2, 1))
        return pred_loc

    def call(self, x):

        # Извлечение признаков
        out = self.conv1(x)
        out = self.conv2(out)
        feat1 = self.pool(out)
        out = self.conv3(feat1)
        out = self.conv4(out)
        feat2 = self.pool(out)
        out = self.conv5(feat2)
        out = self.conv6(out)
        feat3 = self.pool(out)

        # Применение детектора: классификационная часть
        pred_cls1 = self.conv_cls1(feat1)
        pred_cls2 = self.conv_cls2(feat2)
        pred_cls3 = self.conv_cls3(feat3)

        # Применение детектора: локализационная часть
        pred_loc1 = self.conv_loc1(feat1)
        pred_loc2 = self.conv_loc2(feat2)
        pred_loc3 = self.conv_loc3(feat3)

        # Для каждого масштаба переход к тензору размера (batch, num_boxes, num_classes)
        # в тензоре размера (batch, num_boxes, num_classes)
        pred_cls1 = self.reshape_cls(pred_cls1)
        pred_cls2 = self.reshape_cls(pred_cls2)
        pred_cls3 = self.reshape_cls(pred_cls3)

        # Для каждого масштаба получение тензора с координатами всех боксов
        # в тензоре размера (batch, num_boxes, 4)
        pred_loc1 = self.reshape_loc(pred_loc1)
        pred_loc2 = self.reshape_loc(pred_loc2)
        pred_loc3 = self.reshape_loc(pred_loc3)

        # Объединение всех детекций для разнцх масштабов
        pred_cls = tf.concat([pred_cls1, pred_cls2, pred_cls3], axis=1)
        pred_loc = tf.concat([pred_loc1, pred_loc2, pred_loc3], axis=1)

        return pred_cls, pred_loc

model = SSD(num_classes=11, num_def_boxes=3)

### Post-Processing
Сейчас наша SSD модель выдает ответы для всех возможных дефолт-боксов. В полной SSD архитектуре нужны две дополнительные стадии фильтрации: удаление боксов, соответствующих классу "фон" и удаление "дубликатов" с помощью метода Non-Maximum Suppression.

**[ЗАДАНИЕ 2]** Реализуйте первую фильтрацию предсказаний SSD модели -- чтобы остались только боксы, соответствующие объекту (нужно отбросить боксы, соответствующие классу "фон").



In [ ]:
def ssd_filter_predictions(pred_cls, pred_loc, background_class=10, confidence_threshold=0.5):
    """
    Первая фильтрация предсказаний SSD модели.
    Отбрасывает боксы, соответствующие классу "фон".

    Args:
        pred_cls: Тензор классификаций [batch, num_boxes, num_classes]
        pred_loc: Тензор локализаций [batch, num_boxes, 4] (y, x, h, w)
        background_class: Индекс класса "фон" (по умолчанию 10)
        confidence_threshold: Порог уверенности для не-фоновых классов

    Returns:
        Список словарей для каждого изображения в батче:
        {
            'boxes': [N, 4] - координаты боксов (y, x, h, w),
            'scores': [N] - уверенность в предсказании,
            'classes': [N] - предсказанные классы
        }
    """
    batch_size = tf.shape(pred_cls)[0]
    results = []

    for i in range(batch_size):
        # Получаем предсказания для i-го изображения
        cls_i = pred_cls[i]  # [num_boxes, num_classes]
        loc_i = pred_loc[i]  # [num_boxes, 4]

        # Находим максимальную уверенность и соответствующий класс для каждого бокса
        max_scores = tf.reduce_max(cls_i, axis=1)  # [num_boxes]
        pred_classes = tf.argmax(cls_i, axis=1)    # [num_boxes]

        # Фильтруем боксы:
        # 1. Класс не должен быть "фон" (background_class)
        # 2. Уверенность должна быть выше порога
        non_background_mask = tf.not_equal(pred_classes, background_class)
        confidence_mask = tf.greater(max_scores, confidence_threshold)
        mask = tf.logical_and(non_background_mask, confidence_mask)

        # Применяем маску
        filtered_indices = tf.where(mask)

        if tf.shape(filtered_indices)[0] > 0:
            # Собираем отфильтрованные результаты
            filtered_boxes = tf.gather(loc_i, filtered_indices[:, 0])
            filtered_scores = tf.gather(max_scores, filtered_indices[:, 0])
            filtered_classes = tf.gather(pred_classes, filtered_indices[:, 0])

            results.append({
                'boxes': filtered_boxes,
                'scores': filtered_scores,
                'classes': filtered_classes
            })
        else:
            # Нет подходящих боксов
            results.append({
                'boxes': tf.zeros((0, 4), dtype=loc_i.dtype),
                'scores': tf.zeros((0,), dtype=max_scores.dtype),
                'classes': tf.zeros((0,), dtype=pred_classes.dtype)
            })

    return results

In [ ]:
class SSD(tf.keras.Model):
    def __init__(self, num_classes, num_def_boxes):
        super().__init__()
        self.num_classes = num_classes

        # Слои для извлечения признаков

        self.conv1 = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')
        self.conv2 = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')
        self.pool1 = tf.keras.layers.MaxPooling2D((2, 2), (2, 2), padding='same')

        self.conv3 = tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')
        self.conv4 = tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')
        self.pool2 = tf.keras.layers.MaxPooling2D((2, 2), (2, 2), padding='same')

        self.conv5 = tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same')
        self.conv6 = tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same')
        self.conv7 = tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same')
        self.pool3 = tf.keras.layers.MaxPooling2D((2, 2), (2, 2), padding='same')

        self.conv8 = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same')
        self.conv9 = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same')
        self.conv10 = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same')
        self.pool4 = tf.keras.layers.MaxPooling2D((2, 2), (2, 2), padding='same')

        self.conv11 = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same')
        self.conv12 = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same')
        self.conv13 = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same')
        self.pool5 = tf.keras.layers.MaxPooling2D((3, 3), (1, 1), padding='same')

        # Conv6 (атрусовская свертка) - третий масштаб
        self.conv6 = tf.keras.layers.Conv2D(1024, (3, 3), dilation_rate=(6, 6), activation='relu', padding='same')

        # Conv7 - четвертый масштаб
        self.conv7 = tf.keras.layers.Conv2D(1024, (1, 1), activation='relu', padding='same')

        # Conv8 - дополнительный слой для мелких объектов (пятый масштаб)
        self.conv8_1 = tf.keras.layers.Conv2D(256, (1, 1), activation='relu', padding='same')
        self.conv8_2 = tf.keras.layers.Conv2D(512, (3, 3), strides=(2, 2), activation='relu', padding='same')

        # Conv9 - дополнительный слой (шестой масштаб)
        self.conv9_1 = tf.keras.layers.Conv2D(128, (1, 1), activation='relu', padding='same')
        self.conv9_2 = tf.keras.layers.Conv2D(256, (3, 3), strides=(2, 2), activation='relu', padding='same')

        # Conv10 - дополнительный слой (седьмой масштаб)
        self.conv10_1 = tf.keras.layers.Conv2D(128, (1, 1), activation='relu', padding='same')
        self.conv10_2 = tf.keras.layers.Conv2D(256, (3, 3), strides=(1, 1), activation='relu', padding='valid')

        # Conv11 - дополнительный слой (восьмой масштаб)
        self.conv11_1 = tf.keras.layers.Conv2D(128, (1, 1), activation='relu', padding='same')
        self.conv11_2 = tf.keras.layers.Conv2D(256, (3, 3), strides=(1, 1), activation='relu', padding='valid')



        # Классификационные части детекторов (отдельный детектор для каждого масштаба)
        # Для каждого пикселя карт признаков предсказываются
        # распределения вероятностей для всех дефолт боксов
        self.conv_cls1 = tf.keras.layers.Conv2D(num_def_boxes * num_classes, (3, 3), padding='same')
        self.conv_cls2 = tf.keras.layers.Conv2D(num_def_boxes * num_classes, (3, 3), padding='same')
        self.conv_cls3 = tf.keras.layers.Conv2D(num_def_boxes * num_classes, (3, 3), padding='same')
        self.conv_cls4 = tf.keras.layers.Conv2D(num_def_boxes * num_classes, (3, 3), padding='same')

        self.conv_loc1 = tf.keras.layers.Conv2D(num_def_boxes * 4, (3, 3), padding='same')
        self.conv_loc2 = tf.keras.layers.Conv2D(num_def_boxes * 4, (3, 3), padding='same')
        self.conv_loc3 = tf.keras.layers.Conv2D(num_def_boxes * 4, (3, 3), padding='same')
        self.conv_loc4 = tf.keras.layers.Conv2D(num_def_boxes * 4, (3, 3), padding='same')

    # Переход к тензору размера (batch, num_boxes, num_classes)
    # batch - кол-во образцов в батче
    # num_boxes - количество всех боксов для данного масштаба
    # num_classes - количество классов
    def reshape_cls(self, pred_cls):
        batch_size = tf.shape(pred_cls)[0]
        h = tf.shape(pred_cls)[1]
        w = tf.shape(pred_cls)[2]

        # Решейпим: [batch, H, W, num_def_boxes, num_classes] -> [batch, H*W*num_def_boxes, num_classes]
        pred_cls = tf.reshape(pred_cls, [batch_size, h, w, self.num_def_boxes, self.num_classes])
        pred_cls = tf.reshape(pred_cls, [batch_size, -1, self.num_classes])
        return pred_cls

    # Переход к тензору размера (batch, num_boxes, 4)
    # batch - кол-во образцов в батче
    # num_boxes - количество всех боксов для данного масштаба
    def reshape_loc(self, pred_loc):
        batch_size = tf.shape(pred_loc)[0]
        h = tf.shape(pred_loc)[1]
        w = tf.shape(pred_loc)[2]

        # Решейпим: [batch, H, W, num_def_boxes, 4] -> [batch, H*W*num_def_boxes, 4]
        pred_loc = tf.reshape(pred_loc, [batch_size, h, w, self.num_def_boxes, 4])
        pred_loc = tf.reshape(pred_loc, [batch_size, -1, 4])
        return pred_loc

    def call(self, x):

        # === Извлечение признаков VGG-16 ===
        # Block 1
        x = self.conv1_1(x)
        x = self.conv1_2(x)
        x = self.pool1(x)

        # Block 2
        x = self.conv2_1(x)
        x = self.conv2_2(x)
        x = self.pool2(x)

        # Block 3
        x = self.conv3_1(x)
        x = self.conv3_2(x)
        x = self.conv3_3(x)
        x = self.pool3(x)

        # Block 4 (первый масштаб для детекции - conv4_3)
        x = self.conv4_1(x)
        x = self.conv4_2(x)
        feat1 = self.conv4_3(x)  # Первый масштаб
        x = self.pool4(feat1)

        # Block 5
        x = self.conv5_1(x)
        x = self.conv5_2(x)
        x = self.conv5_3(x)
        x = self.pool5(x)

        # Дополнительные слои SSD
        # Conv6 (атрусовская свертка)
        x = self.conv6(x)

        # Conv7 (второй масштаб)
        feat2 = self.conv7(x)

        # Conv8 (третий масштаб)
        x = self.conv8_1(feat2)
        feat3 = self.conv8_2(x)

        # Conv9 (четвертый масштаб)
        x = self.conv9_1(feat3)
        feat4 = self.conv9_2(x)



       # === Применение детекционных головок к 4 масштабам ===
        # 1. feat1 (conv4_3)
        pred_cls1 = self.conv_cls1(feat1)
        pred_loc1 = self.conv_loc1(feat1)

        # 2. feat2 (conv7)
        pred_cls2 = self.conv_cls2(feat2)
        pred_loc2 = self.conv_loc2(feat2)

        # 3. feat3 (conv8_2)
        pred_cls3 = self.conv_cls3(feat3)
        pred_loc3 = self.conv_loc3(feat3)

        # 4. feat4 (conv9_2)
        pred_cls4 = self.conv_cls4(feat4)
        pred_loc4 = self.conv_loc4(feat4)

        # Решейпинг предсказаний
        pred_cls1 = self.reshape_cls(pred_cls1)
        pred_cls2 = self.reshape_cls(pred_cls2)
        pred_cls3 = self.reshape_cls(pred_cls3)
        pred_cls4 = self.reshape_cls(pred_cls4)

        pred_loc1 = self.reshape_loc(pred_loc1)
        pred_loc2 = self.reshape_loc(pred_loc2)
        pred_loc3 = self.reshape_loc(pred_loc3)
        pred_loc4 = self.reshape_loc(pred_loc4)

        # Конкатенация всех масштабов
        pred_cls = tf.concat([pred_cls1, pred_cls2, pred_cls3, pred_cls4], axis=1)
        pred_loc = tf.concat([pred_loc1, pred_loc2, pred_loc3, pred_loc4], axis=1)

        return pred_cls, pred_loc

model = SSD(num_classes=11, num_def_boxes=3)

### Базовая модель
Архитектура SSD может быть реализована поверх любой произвольной модели CNN. Такая модель иногда называется "Базовая модель". Например, если есть CNN модель ResNet-101, то можно реализовать "SSD на основе ResNet-101".

**[ЗАДАНИЕ 3]** Ниже приведена реализация классификационной архитектуры VGG-16 (просто для ознакомления). Реализуйте архитектуру детектирования объектов SSD на основе VGG-16 (по аналогии с примеров в начале). Используйте 4 различных уровня (масштаба) признаков для детекторов (выберите соответствувющие тензоры самостоятельно).

In [ ]:
vgg = tf.keras.Sequential([
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2), (2, 2), padding='same'),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2), (2, 2), padding='same'),
    tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2), (2, 2), padding='same'),
    tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2), (2, 2), padding='same'),
    tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2), (2, 2), padding='same'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(4096, activation='relu'),
    tf.keras.layers.Dense(4096, activation='relu'),
    tf.keras.layers.Dense(1000, activation='softmax'),
])